# ENARES 2024 CRS04 — Stage 03
## 3.4 · Violencia sexual —  + `VS_12M`

Completa el bloque 3.4 en una sola ejecución. identifica al menos una situación de violencia sexual reportada en `C4P248_1:C4P248_16`; `VS_12M` exige además que la situación haya ocurrido en los últimos 12 meses (`C4P248C_i = 1`).


In [ ]:
from google.colab import auth, drive
from google.cloud import bigquery
from pathlib import Path

auth.authenticate_user()
drive.mount("/content/drive")

PROJECT_ID = "enares-2024-crs04"
LOCATION = "US"
EXPECTED_ROWS = 18807
ROOT_DRIVE = Path("/content/drive/MyDrive/ENARES_2024_PROJECT")
SQL_DIR = ROOT_DRIVE / "02SQL"
LOG_DIR = ROOT_DRIVE / "05Resultados" / "logs" / "stage03"
SQL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
A = f"{PROJECT_ID}.enares2024_crs04_analytical.analytical_crs04_adolescents"
print("Tabla:", A)


Mounted at /content/drive
Tabla: enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents


In [ ]:
# 1. Verificar las 32 variables fuente
required_vs12m = []
for i in range(1, 17):
    required_vs12m.extend([f"C4P248_{i}", f"C4P248C_{i}"])

table = client.get_table(A)
columns = {field.name for field in table.schema}
missing_vs12m = sorted(set(required_vs12m) - columns)
if missing_vs12m:
    raise RuntimeError("No se puede crear VS_12M. Faltan: " + ", ".join(missing_vs12m))
if table.num_rows != EXPECTED_ROWS:
    raise RuntimeError(f"La tabla tiene {table.num_rows:,} filas; se esperaban {EXPECTED_ROWS:,}.")
print("Variables fuente verificadas:", len(required_vs12m))
print("Filas verificadas:", table.num_rows)


Variables fuente verificadas: 32
Filas verificadas: 18807


In [ ]:
# 3. Materializar VS_12M con la regla de últimos 12 meses
vs12m_conditions = [
    f"(`C4P248_{i}` = 1 AND `C4P248C_{i}` = 1)"
    for i in range(1, 17)
]
any_vs12m = "\n      OR ".join(vs12m_conditions)
existing_columns = {field.name for field in client.get_table(A).schema}
select_prefix = "* EXCEPT(VS_12M)" if "VS_12M" in existing_columns else "*"

sql_vs12m = f"""
CREATE OR REPLACE TABLE `{A}` AS
SELECT
  {select_prefix},
  CASE
    WHEN (
      {any_vs12m}
    ) THEN 1
    ELSE 0
  END AS VS_12M
FROM `{A}`
""".strip()

sql_path = SQL_DIR / "stage3_34_vs_12m.sql"
sql_path.write_text(sql_vs12m + ";\n", encoding="utf-8")
print(sql_vs12m)
client.query(sql_vs12m, location=LOCATION).result()
print("VS_12M creado correctamente.")
print("SQL guardado en:", sql_path)


CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS
SELECT
  * EXCEPT(VS_12M),
  CASE
    WHEN (
      (`C4P248_1` = 1 AND `C4P248C_1` = 1)
      OR (`C4P248_2` = 1 AND `C4P248C_2` = 1)
      OR (`C4P248_3` = 1 AND `C4P248C_3` = 1)
      OR (`C4P248_4` = 1 AND `C4P248C_4` = 1)
      OR (`C4P248_5` = 1 AND `C4P248C_5` = 1)
      OR (`C4P248_6` = 1 AND `C4P248C_6` = 1)
      OR (`C4P248_7` = 1 AND `C4P248C_7` = 1)
      OR (`C4P248_8` = 1 AND `C4P248C_8` = 1)
      OR (`C4P248_9` = 1 AND `C4P248C_9` = 1)
      OR (`C4P248_10` = 1 AND `C4P248C_10` = 1)
      OR (`C4P248_11` = 1 AND `C4P248C_11` = 1)
      OR (`C4P248_12` = 1 AND `C4P248C_12` = 1)
      OR (`C4P248_13` = 1 AND `C4P248C_13` = 1)
      OR (`C4P248_14` = 1 AND `C4P248C_14` = 1)
      OR (`C4P248_15` = 1 AND `C4P248C_15` = 1)
      OR (`C4P248_16` = 1 AND `C4P248C_16` = 1)
    ) THEN 1
    ELSE 0
  END AS VS_12M
FROM `enares-2024-crs04.enares2024_crs04_analytical.analytical_c

In [ ]:
# 4. Validar dominio y distribución
validation_vs12m = client.query(f"""
SELECT VS_12M, COUNT(*) AS n
FROM `{A}`
GROUP BY VS_12M
ORDER BY VS_12M
""", location=LOCATION).result().to_dataframe()
display(validation_vs12m)
validation_vs12m.to_csv(LOG_DIR / "stage3_34_vs_12m_distribution.csv", index=False)

invalid_vs12m = client.query(f"""
SELECT COUNTIF(VS_12M IS NULL OR VS_12M NOT IN (0,1)) AS invalid_values
FROM `{A}`
""", location=LOCATION).result().to_dataframe()
display(invalid_vs12m)
if int(invalid_vs12m.iloc[0]["invalid_values"]) != 0:
    raise RuntimeError("VS_12M contiene NULL o valores fuera de 0/1.")
print("Dominio VS_12M aprobado: 0/1 sin NULL.")


,VS_12M,n
0,0,15378
1,1,3429


,invalid_values
0,0


Dominio VS_12M aprobado: 0/1 sin NULL.


In [ ]:
# 5. Comprobar equivalencia exacta contra la regla SPSS
consistency_sql = f"""
SELECT
  COUNT(*) AS total_rows,
  COUNTIF(
    VS_12M != CASE WHEN ({any_vs12m}) THEN 1 ELSE 0 END
  ) AS inconsistent_rows
FROM `{A}`
"""
consistency_vs12m = client.query(consistency_sql, location=LOCATION).result().to_dataframe()
display(consistency_vs12m)
consistency_vs12m.to_csv(LOG_DIR / "stage3_34_vs_12m_consistency.csv", index=False)
if int(consistency_vs12m.iloc[0]["inconsistent_rows"]) != 0:
    raise RuntimeError("VS_12M no coincide exactamente con la regla SPSS.")
if int(consistency_vs12m.iloc[0]["total_rows"]) != EXPECTED_ROWS:
    raise RuntimeError("El número de filas cambió durante la creación de VS_12M.")
print("VS_12M validado exactamente contra la regla SPSS.")


,total_rows,inconsistent_rows
0,18807,0


VS_12M validado exactamente contra la regla SPSS.


In [ ]:
# 6. Validación conjunta del bloque 3.4
validation_34 = client.query(f"""
SELECT
  COUNT(*) AS total_rows,
  COUNTIF(INDICADOR_8_3_6 NOT IN (0,1) OR INDICADOR_8_3_6 IS NULL) AS invalid_indicator,
  COUNTIF(VS_12M NOT IN (0,1) OR VS_12M IS NULL) AS invalid_vs12m,
  COUNTIF(VS_12M = 1 AND INDICADOR_8_3_6 != 1) AS recent_without_any_vs
FROM `{A}`
""", location=LOCATION).result().to_dataframe()

display(validation_34)
validation_34.to_csv(LOG_DIR / "stage3_34_final_validation.csv", index=False)

r = validation_34.iloc[0]
if int(r["total_rows"]) != EXPECTED_ROWS:
    raise RuntimeError("El universo cambió durante Stage 3.4.")
if int(r["invalid_indicator"]) != 0:
    raise RuntimeError("INDICADOR_8_3_6 tiene valores fuera de 0/1.")
if int(r["invalid_vs12m"]) != 0:
    raise RuntimeError("VS_12M tiene valores fuera de 0/1.")
if int(r["recent_without_any_vs"]) != 0:
    raise RuntimeError("Hay VS_12M=1 con INDICADOR_8_3_6 distinto de 1.")

print("PASS Stage 3.4: INDICADOR_8_3_6 y VS_12M materializados y validados.")


,total_rows,invalid_indicator,invalid_vs12m,recent_without_any_vs
0,18807,0,0,0


PASS Stage 3.4: INDICADOR_8_3_6 y VS_12M materializados y validados.
